# MOESCORE Expert 3: M2D-CLAP fine-tuning

This notebook runs the dedicated Expert 3 trainer with a **physical training batch of 8** and no gradient accumulation. Outputs are written to Google Drive. The smoke-test cell runs a real forward, backward, and optimizer step before training.

After mounting Drive, edit the three path variables below to match your Drive folders. `DATA_ROOT` must contain `meta_data/` and `wav/`; the checkpoint is the untouched M2D-CLAP 2025 `checkpoint-30.pth`.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, json
drive.mount('/content/drive')

# Replace these placeholders with folders that exist in your MyDrive.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/EDIT_THIS_EXPERT3_PROJECT_FOLDER')
DATA_ROOT = Path('/content/drive/MyDrive/EDIT_THIS_DATASET_ROOT')
M2D_CHECKPOINT_PATH = Path('/content/drive/MyDrive/EDIT_THIS_CHECKPOINT_PATH/checkpoint-30.pth')
if any('EDIT_THIS' in str(p) for p in [DRIVE_PROJECT_DIR, DATA_ROOT, M2D_CHECKPOINT_PATH]):
    raise ValueError('Edit DRIVE_PROJECT_DIR, DATA_ROOT, and M2D_CHECKPOINT_PATH to the actual Drive paths.')
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'runs/expert3_moescore'
CONFIG_PATH = DRIVE_PROJECT_DIR / 'configs/expert3_moescore.json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project:', DRIVE_PROJECT_DIR)
print('Dataset:', DATA_ROOT)
print('M2D checkpoint:', M2D_CHECKPOINT_PATH)
print('Output (persistent on Drive):', OUTPUT_DIR)


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable Runtime → Change runtime type → T4 GPU, then rerun this cell.')
gpu_name = torch.cuda.get_device_name(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print('GPU:', gpu_name)
print(f'VRAM total/free: {total_bytes / 2**30:.2f} / {free_bytes / 2**30:.2f} GiB')
print('PyTorch:', torch.__version__, '| CUDA build:', torch.version.cuda)
if 'T4' not in gpu_name:
    print('Note: Colab assigned a different CUDA GPU; batch size remains fixed at 8.')


In [ ]:
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(CONFIG_PATH)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(DRIVE_PROJECT_DIR / 'requirements-expert3.txt')], check=True)
print('Installed Expert 3 dependencies. PyTorch was not included in the requirements file.')


## Validate paths and optionally stage input assets

Staging data/checkpoint under `/content` can speed up repeated reads. It is optional and needs enough temporary disk space. Training outputs remain on Drive.


In [ ]:
required_project_files = [
    'train_expert3.py', 'inference_expert3.py', 'models/moescore_expert3.py',
    'datasets/moescore_expert3_dataset.py', 'utils/expert3_checkpoint.py',
    'configs/expert3_moescore.json', 'requirements-expert3.txt',
    'losses/loss_function.py', 'models/m2d/vendor/portable_m2d.py',
    'models/m2d/encoder.py',
]
missing = [name for name in required_project_files if not (DRIVE_PROJECT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing code files in project folder: {missing}')
if not M2D_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(M2D_CHECKPOINT_PATH)
for name in ['meta_data/train_average.csv', 'meta_data/validation_average.csv',
             'meta_data/test_average.csv', 'wav/train', 'wav/validation', 'wav/test']:
    if not (DATA_ROOT / name).exists():
        raise FileNotFoundError(DATA_ROOT / name)

COPY_DATA_TO_LOCAL = True
COPY_CHECKPOINT_TO_LOCAL = True
required_local_bytes = M2D_CHECKPOINT_PATH.stat().st_size if COPY_CHECKPOINT_TO_LOCAL else 0
if COPY_DATA_TO_LOCAL:
    required_local_bytes += sum(Path(directory, filename).stat().st_size
                               for directory, _, filenames in os.walk(DATA_ROOT)
                               for filename in filenames)
free_local_bytes = shutil.disk_usage('/content').free
if free_local_bytes < required_local_bytes * 1.1:
    raise OSError(f'Not enough /content space: need {required_local_bytes / 2**30:.1f} GiB, available {free_local_bytes / 2**30:.1f} GiB; disable staging to continue.')
if COPY_DATA_TO_LOCAL:
    staged_data = Path('/content/expert3_dataset')
    stage_marker = staged_data / '.copy_complete'
    if not stage_marker.is_file():
        shutil.copytree(DATA_ROOT, staged_data, dirs_exist_ok=True)
        stage_marker.touch()
    ACTIVE_DATA_ROOT = staged_data
else:
    ACTIVE_DATA_ROOT = DATA_ROOT
if COPY_CHECKPOINT_TO_LOCAL:
    staged_checkpoint = Path('/content/m2d_checkpoint-30.pth')
    if not staged_checkpoint.is_file() or staged_checkpoint.stat().st_size != M2D_CHECKPOINT_PATH.stat().st_size:
        shutil.copy2(M2D_CHECKPOINT_PATH, staged_checkpoint)
    ACTIVE_CHECKPOINT = staged_checkpoint
else:
    ACTIVE_CHECKPOINT = M2D_CHECKPOINT_PATH
CACHE_DIR = Path('/content/expert3_hf_cache')
print('Active dataset:', ACTIVE_DATA_ROOT)
print('Active checkpoint:', ACTIVE_CHECKPOINT)


In [ ]:
COMMON = [
    '--config', str(CONFIG_PATH),
    '--data-root', str(ACTIVE_DATA_ROOT),
    '--m2d-checkpoint', str(ACTIVE_CHECKPOINT),
    '--output-dir', str(OUTPUT_DIR),
    '--cache-dir', str(CACHE_DIR),
    '--device', 'cuda',
]
subprocess.run([sys.executable, str(DRIVE_PROJECT_DIR / 'train_expert3.py'),
                *COMMON, '--check-data'], cwd=DRIVE_PROJECT_DIR, check=True)


In [ ]:
# Smoke-test does one real batch of 8, including backward and an optimizer update.
subprocess.run([sys.executable, str(DRIVE_PROJECT_DIR / 'train_expert3.py'),
                *COMMON, '--smoke-test'], cwd=DRIVE_PROJECT_DIR, check=True)


## Train or resume

Set `RESUME_CHECKPOINT` to a previous run's `last_checkpoint.pt` to resume. Leave it `None` for a new run. A checkpoint is saved after each completed epoch. The initial configuration enforces batch size 8, `drop_last=true`, and accumulation 1; an OOM stops with a diagnostic rather than changing the objective.


In [ ]:
RESUME_CHECKPOINT = None  # Example: Path('/content/drive/MyDrive/.../last_checkpoint.pt')
train_command = [sys.executable, str(DRIVE_PROJECT_DIR / 'train_expert3.py'), *COMMON]
if RESUME_CHECKPOINT is not None:
    if not Path(RESUME_CHECKPOINT).is_file():
        raise FileNotFoundError(RESUME_CHECKPOINT)
    train_command += ['--resume', str(RESUME_CHECKPOINT)]
subprocess.run(train_command, cwd=DRIVE_PROJECT_DIR, check=True)
if RESUME_CHECKPOINT is not None:
    RUN_DIR = Path(RESUME_CHECKPOINT).parent
else:
    RUN_DIR = max((p for p in OUTPUT_DIR.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime)
print('Run artifacts on Drive:', RUN_DIR)


In [ ]:
best_checkpoint = RUN_DIR / 'best_model.pt'
if not best_checkpoint.is_file():
    raise FileNotFoundError(best_checkpoint)
for split in ['validation', 'test']:
    subprocess.run([sys.executable, str(DRIVE_PROJECT_DIR / 'inference_expert3.py'),
                    '--config', str(RUN_DIR / 'config.json'),
                    '--checkpoint', str(best_checkpoint), '--dataset', split,
                    '--data-root', str(ACTIVE_DATA_ROOT), '--m2d-checkpoint', str(ACTIVE_CHECKPOINT),
                    '--cache-dir', str(CACHE_DIR), '--device', 'cuda'],
                   cwd=DRIVE_PROJECT_DIR, check=True)
print('Best checkpoint:', best_checkpoint)
print('Resume checkpoint:', RUN_DIR / 'last_checkpoint.pt')
print('Training log:', RUN_DIR / 'training_log.txt')
print('Validation selection metrics:', RUN_DIR / 'best_metrics.json')
